# FireFusion — Fire–Climate Join Documentation

**Project:** FireFusion  
**Stream:** Data Engineering  
**Task:** Fire–Climate Join Documentation  
**Author:** Archit Chandna  
**Student ID:** 224030033  

## Document Purpose

This document describes the existing FireFusion fire–climate enrichment workflow implemented in `data-engineering/pipelines/fire_climate_enrichment/enrich_fire_climate.py`.

The current pipeline enriches fire records with two environmental signals:

1. **Local weather** from the realtime weather pipeline.
2. **ENSO / Oceanic Niño Index (ONI)** from the NOAA CPC ENSO pipeline.

It documents the existing join logic, inputs, outputs, temporal matching behaviour, edge cases and assumptions, and provides a reusable template for future climate signals such as the Indian Ocean Dipole (IOD).

> **Scope:** Existing implementation details and proposed future improvements are deliberately separated. The IOD section is a design template and does not claim that IOD enrichment is already implemented.

## 1. Pipeline Overview

```text
sample_fires.csv
       |
       | fire_datetime -> record_year_month
       v
+---------------------------+
| Local Weather Enrichment  |
| Key: location_id          |
| Join: LEFT                |
+---------------------------+
       |
       v
Weather-enriched fire records
       |
       | record_year_month -> enso_match_date
       v
+---------------------------+
| ENSO / ONI Enrichment     |
| Temporal ASOF join        |
| Direction: backward       |
+---------------------------+
       |
       v
sample_fires_dual_climate_enriched.csv
```

The implementation therefore combines a **location-based weather join** with a **monthly temporal climate join**.

## 2. Inputs

### 2.1 Fire records

Configured input:

`data-engineering/pipelines/fire_climate_enrichment/sample_fires.csv`

The existing script requires the fire data to provide at least:

- `fire_datetime` — timestamp used to derive the monthly climate key.
- `location_id` — identifier used for local weather enrichment.

### 2.2 Realtime weather

Configured input:

`data-engineering/pipelines/realtime_weather/output/realtime_weather_validated.csv`

Fields selected by the enrichment:

- `location_id`
- `temperature_c`
- `wind_speed_kmh`
- `relative_humidity`
- `source_system`

### 2.3 ENSO / ONI

Configured input in the reviewed implementation:

`data-engineering/data/processed/noaa_cpc_enso_oni_processed_20260906.csv`

Fields used for enrichment:

- `record_year_month`
- `oni_anomaly`
- `enso_phase`
- `oni_lag6m`

The ENSO processor also produces `enso_id`, `time_id`, `datetime_record` and `original_source`.

## 3. Fire Data Preparation

The fire loader reads the CSV, parses the fire timestamp and derives a monthly key:

```python
df = pd.read_csv(FIRE_FILE)
df["fire_datetime"] = pd.to_datetime(df["fire_datetime"])
df["record_year_month"] = df["fire_datetime"].dt.strftime("%Y-%m")
```

Example:

```text
fire_datetime       = 2026-09-09 13:00:00
record_year_month   = 2026-09
```

ENSO is a monthly signal, so converting a fire timestamp into `YYYY-MM` gives the enrichment pipeline a common temporal representation for matching.

## 4. Local Weather Join

### 4.1 Fields selected

```python
weather_fields = weather[
    [
        "location_id",
        "temperature_c",
        "wind_speed_kmh",
        "relative_humidity",
        "source_system",
    ]
]
```

### 4.2 Join key and type

```text
Fire.location_id = Weather.location_id
```

Existing implementation:

```python
return fires.merge(
    weather_fields,
    on="location_id",
    how="left"
)
```

This is a **left join**. Fire records remain the base population. A matching location adds weather attributes; when no matching weather row exists, the fire remains and the weather fields are missing.

### 4.3 Current behaviour to note

The weather join uses only `location_id`. A timestamp is not part of the weather merge condition. If the weather input contains multiple rows with the same `location_id`, a normal pandas merge can produce multiple output rows for one fire record. The current enrichment function does not explicitly deduplicate these rows or perform nearest-time weather matching.

## 5. ENSO Processing

ENSO ingestion and transformation are implemented in `data-engineering/pipelines/enso/fetch_enso.py`.

### 5.1 Source and standardisation

The processor retrieves NOAA CPC Oceanic Niño Index data, preserves a raw copy, reads the space-delimited source and maps seasonal labels to month numbers.

| Season | Month | Season | Month |
|---|---:|---|---:|
| DJF | 01 | JJA | 07 |
| JFM | 02 | JAS | 08 |
| FMA | 03 | ASO | 09 |
| MAM | 04 | SON | 10 |
| AMJ | 05 | OND | 11 |
| MJJ | 06 | NDJ | 12 |

It creates:

- `record_year_month`
- `datetime_record`
- `time_id`
- `oni_anomaly`
- `enso_phase`
- `oni_lag6m`
- `original_source`
- `enso_id`

### 5.2 ENSO phase

The implemented thresholds are:

```text
oni_anomaly >=  0.5  -> El Nino
oni_anomaly <= -0.5  -> La Nina
otherwise            -> Neutral
```

### 5.3 Six-month lag feature

```python
df["oni_lag6m"] = df["oni_anomaly"].shift(6)
```

### 5.4 Lineage

The processor assigns:

```text
original_source = NOAA_CPC_ONI
```

This preserves the source identity in the processed ENSO data.

## 6. ENSO Temporal Join Logic

### 6.1 Build a month-start matching key

For fires:

```python
fires["enso_match_date"] = pd.to_datetime(
    fires["record_year_month"] + "-01"
)
```

For ENSO:

```python
enso["enso_match_date"] = pd.to_datetime(
    enso["record_year_month"] + "-01"
)
```

Example:

```text
record_year_month = 2026-09
enso_match_date   = 2026-09-01
```

### 6.2 Select and sort ENSO fields

```python
enso_fields = enso[
    [
        "enso_match_date",
        "oni_anomaly",
        "enso_phase",
        "oni_lag6m",
    ]
].sort_values("enso_match_date")

fires = fires.sort_values("enso_match_date")
```

### 6.3 Backward ASOF join

```python
enriched = pd.merge_asof(
    fires,
    enso_fields,
    on="enso_match_date",
    direction="backward"
)
```

For each fire record, `direction="backward"` selects the most recent ENSO record whose matching date is **less than or equal to** the fire matching date.

```text
Exact month available
Fire Sep -> ENSO Sep -> use Sep

Exact month unavailable
ENSO Jul -> ENSO Aug ---------> ENSO Oct
                    ^
                    |
                 Fire Sep
                 uses Aug
```

The existing README also states that when an exact ENSO month is unavailable, the latest available ENSO record before the fire date is used.

## 7. End-to-End Execution Sequence

The existing `main()` function executes:

```python
fires = load_sample_fires()
weather = load_weather()
enso = load_enso()

enriched = enrich_with_weather(fires, weather)
enriched = enrich_with_enso(enriched, enso)

enriched.to_csv(OUTPUT_FILE, index=False)
```

The workflow is:

1. Load fire records.
2. Parse `fire_datetime`.
3. Create `record_year_month`.
4. Load validated realtime weather.
5. Left-join weather using `location_id`.
6. Load processed ENSO data.
7. Convert monthly keys into `enso_match_date`.
8. Sort the fire and ENSO data by the matching date.
9. Perform a backward temporal ASOF join.
10. Add ENSO fields.
11. Save the enriched dataset.

Configured output:

`data-engineering/pipelines/fire_climate_enrichment/sample_fires_dual_climate_enriched.csv`

## 8. Output Contract

The enriched output retains the fire information and adds environmental context.

### Weather fields added

- `temperature_c`
- `wind_speed_kmh`
- `relative_humidity`
- `source_system`

### ENSO fields added

- `enso_match_date`
- `oni_anomaly`
- `enso_phase`
- `oni_lag6m`

Conceptually:

```text
Fire Event
   |
   +-- original fire attributes
   |
   +-- local weather context
   |     +-- temperature_c
   |     +-- wind_speed_kmh
   |     +-- relative_humidity
   |
   +-- ENSO climate context
         +-- oni_anomaly
         +-- enso_phase
         +-- oni_lag6m
```

## 9. Edge Cases, Assumptions and Limitations

### Missing weather match
The left join preserves the fire record. Weather fields remain missing.

### Missing exact ENSO month
The backward ASOF join uses the latest ENSO record at or before the fire month.

### Fire earlier than available ENSO history
If no ENSO record exists at or before the fire matching date, there is no backward candidate and the ENSO fields remain unmatched.

### Invalid fire datetime
The implementation calls `pd.to_datetime(df["fire_datetime"])` without `errors="coerce"` or explicit exception handling. Invalid values can therefore cause parsing to fail.

### Duplicate weather locations
Multiple weather rows sharing a `location_id` can multiply rows during the weather merge.

### No ENSO match tolerance
The current `merge_asof` does not specify `tolerance`, so no maximum temporal gap is enforced by this join.

### Date-stamped ENSO input
The reviewed code directly references `noaa_cpc_enso_oni_processed_20260906.csv`. A future production version could resolve the latest validated dataset through configuration or controlled discovery.

### Post-join validation
The reviewed enrichment script does not contain explicit row-count, unmatched-rate, duplicate or temporal-gap validation after the joins. These are recommendations for future strengthening rather than existing functionality.

## 10. Template for Future Climate Signals

The ENSO implementation provides a reusable structure for additional temporal climate indicators.

A future signal should document:

1. Authoritative source and lineage.
2. Temporal resolution.
3. Standard temporal key.
4. Numerical signal fields.
5. Classification/phase fields where relevant.
6. Lag features where relevant.
7. Matching direction.
8. Match tolerance if required.
9. Missing-match behaviour.
10. Validation rules.

### Generic temporal enrichment template

```python
def enrich_with_climate_signal(fires, signal):
    fires = fires.copy()
    signal = signal.copy()

    fires["signal_match_date"] = pd.to_datetime(
        fires["record_year_month"] + "-01"
    )

    signal["signal_match_date"] = pd.to_datetime(
        signal["record_year_month"] + "-01"
    )

    signal_fields = signal[
        [
            "signal_match_date",
            "signal_value",
            "signal_phase",
        ]
    ].sort_values("signal_match_date")

    fires = fires.sort_values("signal_match_date")

    return pd.merge_asof(
        fires,
        signal_fields,
        on="signal_match_date",
        direction="backward"
    )
```

> This is a proposed documentation template derived from the structure of the existing ENSO implementation. It is not existing FireFusion production code.

## 11. IOD Extension Template

The current sprint identifies the Indian Ocean Dipole (IOD) as a future climate signal. Once the team's IOD processor and final schema are available, an enrichment function can follow the established temporal pattern where that pattern is appropriate.

```python
def enrich_with_iod(fires, iod):
    fires = fires.copy()
    iod = iod.copy()

    fires["iod_match_date"] = pd.to_datetime(
        fires["record_year_month"] + "-01"
    )

    iod["iod_match_date"] = pd.to_datetime(
        iod["record_year_month"] + "-01"
    )

    iod_fields = iod[
        [
            "iod_match_date",
            "iod_value",
            "iod_phase",
        ]
    ].sort_values("iod_match_date")

    fires = fires.sort_values("iod_match_date")

    return pd.merge_asof(
        fires,
        iod_fields,
        on="iod_match_date",
        direction="backward"
    )
```

`iod_value` and `iod_phase` are **placeholders**. The final implementation must use the actual IOD schema and must respect the temporal semantics and validation rules defined by the IOD processor.

The IOD integration should not automatically inherit ENSO assumptions if the final IOD dataset uses a different temporal resolution or matching rule.

```text
Fire
 |
 +--> Local Weather
 |
 +--> ENSO
 |
 +--> IOD
 |
 v
Multi-signal Fire–Climate Dataset
```

## 12. Future Climate Signal Contract

A small documented contract can make future integrations consistent.

| Contract item | Required documentation |
|---|---|
| Signal name | Human-readable signal name |
| Source | Authoritative source |
| Temporal resolution | Monthly, daily, etc. |
| Standard time field | Field used for matching |
| Value fields | Numerical indicators |
| Category fields | Phase/classification if applicable |
| Lag fields | Historical features if applicable |
| Match strategy | Exact, backward, nearest, etc. |
| Match tolerance | Maximum acceptable gap if required |
| Missing-match behaviour | Expected behaviour when no valid match exists |
| Lineage | Source/origin identifier |

This prevents future climate signals from being integrated using undocumented assumptions.

## 13. Recommended Validation Checklist

These checks are recommended for future hardening of the enrichment pipeline.

### Before joining
- Confirm all required columns exist.
- Validate `fire_datetime`.
- Validate `location_id`.
- Validate climate date parsing.
- Check expected uniqueness of join keys.
- Sort temporal climate records before ASOF matching.

### After weather enrichment
- Compare row counts before and after the join.
- Count unmatched weather records.
- Detect row multiplication from duplicate weather `location_id` values.

### After climate enrichment
- Count unmatched climate records.
- Validate allowed phase/category values.
- Check null rates in signal fields.
- Measure temporal gaps where relevant.
- Confirm that no fire records were unintentionally removed.

These are recommendations and are not represented as checks already implemented by `enrich_fire_climate.py`.

## 14. How to Run the Existing Enrichment

From the repository root:

```bash
python data-engineering/pipelines/fire_climate_enrichment/enrich_fire_climate.py
```

The existing script loads its configured fire, weather and ENSO files, performs both enrichment stages and writes the final CSV.

Before running, the configured input files must exist at the paths expected by the implementation.

## 15. Implementation References

This documentation was prepared by reviewing the existing FireFusion implementation on the `feature/fire-climate-enrichment` branch.

**Primary enrichment implementation**  
`data-engineering/pipelines/fire_climate_enrichment/enrich_fire_climate.py`

**Fire Climate Enrichment README**  
`data-engineering/pipelines/fire_climate_enrichment/README.md`

**ENSO ingestion and processing**  
`data-engineering/pipelines/enso/fetch_enso.py`

**ENSO dataset documentation path**  
`data-engineering/data/docs/noaa_cpc_enso_oni.md`

## 16. Summary

The current FireFusion fire–climate enrichment pipeline uses two distinct strategies:

- **Local weather:** `location_id`-based pandas **left join**.
- **ENSO:** monthly temporal `merge_asof` with **backward** matching.

The fire dataset remains the base dataset while receiving local weather attributes and broader ENSO climate context. If an exact ENSO month is unavailable, the implementation uses the most recent earlier ENSO record.

The same architecture provides a clear starting point for future climate signals such as IOD, provided that each new signal's schema, temporal resolution and matching semantics are explicitly documented rather than assumed. A standard signal contract and post-join validation layer would further improve consistency, traceability and maintainability.

In [ ]:
# Optional reference cell.
# This cell does not execute the enrichment or modify project data.

from pathlib import Path

references = {
    "Fire-climate enrichment": Path(
        "data-engineering/pipelines/fire_climate_enrichment/enrich_fire_climate.py"
    ),
    "Enrichment README": Path(
        "data-engineering/pipelines/fire_climate_enrichment/README.md"
    ),
    "ENSO processor": Path(
        "data-engineering/pipelines/enso/fetch_enso.py"
    ),
    "ENSO documentation": Path(
        "data-engineering/data/docs/noaa_cpc_enso_oni.md"
    ),
}

for name, path in references.items():
    print(f"{name}: {path}")